# 🌍 AI Travel Planner using Gradio

**Week 2 — Day 2 Assignment**

This notebook demonstrates how to build an interactive web application using **Gradio** that leverages **Large Language Models (LLMs)** to generate personalized travel itineraries.

The application allows users to input travel preferences such as destination, number of days, budget, and travel style, and generates a structured travel plan using an AI model accessed through **OpenRouter**.

## Objective

The goal of this assignment is to:

- Learn how to build an interactive UI using **Gradio**
- Connect a frontend interface to a **Large Language Model**
- Allow users to dynamically select different models
- Generate structured AI responses formatted using **Markdown**
- Stream responses to create a smooth user experience

The final application runs locally in the browser and can also be deployed as a public web app.

## Application Features

This AI Travel Planner includes the following features:

- 🌍 **Destination-based itinerary generation**
- 🎛 **Interactive UI built with Gradio**
- 🤖 **Model selection dropdown using OpenRouter models**
- ✨ **Streaming responses for better user experience**
- 📝 **Markdown formatted travel itineraries**
- 📚 **Preloaded example prompts for quick testing**

The application demonstrates how LLM-powered tools can be turned into practical web interfaces.

## Technology Stack

The following tools and libraries are used in this project:

- **Python** – Core programming language
- **Gradio** – Used to build the interactive web interface
- **OpenRouter API** – Provides access to multiple LLMs
- **Markdown Rendering** – Used for structured output formatting

These tools together allow rapid prototyping of AI-powered applications.

## Application Workflow

The system works in the following steps:

1. The user enters travel details such as destination, number of days, budget, and travel style.
2. The user selects a language model from the available options.
3. The application constructs a prompt using the provided information.
4. The prompt is sent to the selected AI model via the OpenRouter API.
5. The model generates a detailed travel itinerary.
6. The response is streamed back and displayed in **Markdown format** within the Gradio interface.

## User Inputs

The Gradio interface collects the following inputs from the user:

- **Destination** – The location the user wants to visit
- **Number of Days** – Duration of the trip
- **Budget** – Low, Medium, or Luxury
- **Travel Style** – Adventure, Culture, Family, Backpacking, etc.
- **Interests** – Specific preferences such as food, history, nature
- **Model Selection** – Allows switching between available LLMs

## Output

The AI generates a structured travel itinerary formatted in Markdown.

The output typically includes:

- Trip overview
- Day-by-day travel plan
- Recommended food and attractions
- Budget breakdown
- Travel tips

The Markdown formatting ensures the output is readable and visually structured within the interface.

## Running the Application

When the final cell in this notebook is executed, the Gradio application launches locally and can be accessed in the browser.

Example:

http://127.0.0.1:7860

The interface allows users to interact with the travel planner and generate itineraries dynamically.

## Conclusion

This project demonstrates how **Gradio can be used to rapidly build AI-powered web interfaces**.

By combining a user-friendly UI with powerful language models, we can create useful applications such as travel planners, assistants, and content generators.

The same application can also be deployed to platforms such as **Hugging Face Spaces**, allowing anyone to access the tool through a public web link.

In [ ]:
%pip install --upgrade gradio openai requests python-dotenv

In [2]:
# Import required libraries
import gradio as gr
import os
from openai import OpenAI

In [4]:
# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv()

# Get OpenRouter API key
api_key = os.getenv("OPENROUTER_API_KEY")

# Initialize OpenRouter client
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [5]:
# Travel planner function that calls the LLM and streams output

def generate_travel_plan(destination, days, budget, style, interests, model):

    prompt = f"""
You are an expert travel planner.

Create a detailed travel itinerary in MARKDOWN format.

Destination: {destination}
Trip length: {days} days
Budget level: {budget}
Travel style: {style}
User interests: {interests}

Structure the response like this:

# 🌍 Travel Plan for {destination}

## Overview
Brief description of the destination and travel vibe.

## Day 1
Activities and places.

## Day 2
Activities and places.

Continue for all {days} days.

## 🍜 Food to Try
Local food recommendations.

## 💰 Budget Tips
Suggestions to optimize spending.

## ✈️ Travel Tips
Important travel advice.
"""

    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    response = ""

    for chunk in completion:
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response

In [6]:
# Supported models via OpenRouter

models = [
    "openai/gpt-4o-mini",
    "anthropic/claude-3-haiku",
    "meta-llama/llama-3-70b-instruct",
    "mistralai/mistral-7b-instruct"
]

In [ ]:
# Build the Gradio UI

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 AI Travel Planner")
    gr.Markdown("Plan your perfect trip using AI")

    with gr.Row():

        with gr.Column():

            destination = gr.Textbox(label="Destination")

            days = gr.Slider(
                minimum=1,
                maximum=14,
                value=5,
                label="Number of days"
            )

            budget = gr.Dropdown(
                ["low", "medium", "luxury"],
                label="Budget"
            )

            style = gr.Dropdown(
                ["backpacking", "family", "luxury", "adventure", "culture"],
                label="Travel style"
            )

            interests = gr.Textbox(label="Interests")

            model = gr.Dropdown(
                models,
                label="Model",
                value=models[0]
            )

            generate_button = gr.Button("Generate Travel Plan")

        with gr.Column():

            output = gr.Markdown()

    gr.Examples(
        examples=[
            ["Paris",5,"medium","culture","food","openai/gpt-4o-mini"],
            ["Tokyo",7,"luxury","adventure","anime","openai/gpt-4o-mini"],
            ["Kerala",4,"medium","family","nature","openai/gpt-4o-mini"]
        ],
        inputs=[destination,days,budget,style,interests,model]
    )

    generate_button.click(
        generate_travel_plan,
        inputs=[destination,days,budget,style,interests,model],
        outputs=output
    )

C:\Users\Suman\AppData\Local\Temp\ipykernel_24348\2850064583.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
